# Lab 1: Tokenisation with NumPy

**Course:** AAE-PT02M1 | **Monday, Lesson 4 of 10** | *How LLMs Process Language: Tokenisation*

---

## The story so far

Olu's manager asks: *"Do you actually understand how the AI reads a patient's message?"* Olu realises she needs to understand the internal mechanics. Not to build a model, but to **debug problems, optimise costs, and explain system behaviour to stakeholders**.

## What this notebook does

This notebook takes the lab code from the lesson **exactly as it is written** and explains it, line by line, so you can see what each part does and why. The code cells below are the lesson's code, only split into small pieces so each piece can be explained on its own. Nothing has been added to the code.

## Where this fits

Every time a patient sends a message to AfyaPlus, the LLM processes it in three stages: **Tokenisation**, Context Understanding, and Prediction. This lab covers the first.

An LLM cannot read text the way humans do. **It only processes numbers.** Before anything else, the system must:

1. Split raw text into small pieces called **tokens** (words, parts of words, or symbols).
2. Map each token to a unique **integer ID**.

For example: `'My chest hurts'` becomes `['my', 'chest', 'hurts']`, then `[42, 1087, 3291]`.

## Why this matters: cost

Token count directly affects cost. OpenAI charges per token. A 200-word patient message might be 250+ tokens. At AfyaPlus's scale of thousands of daily messages, even small token savings add up to significant cost reductions.

> **Engineering reality check**
> In real production AI engineering, NumPy is never used to split strings or handle text cleaning. Specialised tools such as Hugging Face's `tokenizers` or OpenAI's `tiktoken` do this using advanced algorithms like **Byte-Pair Encoding (BPE)**. NumPy is used here purely as a simplified training script to show how text becomes a matrix of numbers. In production you will use a library like `tiktoken` to count tokens *before* sending requests to the API, so you can estimate cost precisely.

## Setup (first time only)

Run these once in a terminal (not in this notebook):

```bash
python -m venv venv
source venv/bin/activate        # Windows: venv\Scripts\activate
pip install numpy
```

If `python` is not found, try `python3`. In Google Colab, NumPy is already installed and you can skip this.

## How to use this notebook

Run the cells **in order** (`Shift + Enter`). Read the explanation above each cell, run it, then compare the output with what the explanation says you should see.

---

## Build it step by step

The lesson builds the lab in five steps. We follow the same five steps.

| Step | What happens |
|---|---|
| 1 | Import NumPy |
| 2 | Clean the text |
| 3 | Split into tokens |
| 4 | Build a vocabulary |
| 5 | Print and inspect |

### Step 1: Import NumPy

We use NumPy because production tokenisers work with **arrays of numbers**, not plain Python lists. The import line gives us the `np` toolkit. The `as np` part is a convention, so you will see `np.` in every NumPy tutorial and codebase.

In [ ]:
import numpy as np

### Steps 2, 3 and 4: the tokeniser function

The whole tokeniser lives inside one function, `simple_numpy_tokenizer`. A **function** is a named, reusable block of code. It takes an input (here `text`), does some work, and gives back results with `return`.

Run the cell first, then read the line-by-line explanation below it. Running this cell does not print anything: it only *defines* the function so we can use it in the next cell.

In [ ]:
def simple_numpy_tokenizer(text):
    # 1. Basic cleaning and splitting
    # We convert to lowercase and remove simple punctuation
    clean_text = text.lower().replace(".", "").replace("?", "")
    tokens = clean_text.split()

    # 2. Convert to NumPy array
    token_array = np.array(tokens)

    # 3. Create a vocabulary (unique tokens)
    # This represents the internal 'dictionary' our system recognises
    vocab, inverse_indices = np.unique(token_array, return_inverse=True)

    return token_array, vocab, inverse_indices

### Line-by-line explanation

**`def simple_numpy_tokenizer(text):`**
Defines the function. `text` is the **parameter**: whatever string you pass in when you call the function.

---

**Step 2: Clean the text.**

**`clean_text = text.lower().replace(".", "").replace("?", "")`**

This is three string operations chained together, read left to right:

- `text.lower()` returns a new string with every letter in lowercase. After this, `Chest` and `chest` are the same, so they will become the same token. (Python strings cannot be changed in place, so each method returns a *new* string and the original is untouched.)
- `.replace(".", "")` returns a new string in which every full stop is swapped for an empty string, which means **deleted**.
- `.replace("?", "")` does the same for every question mark.

So `'My chest hurts. Is it a heart attack?'` becomes `'my chest hurts is it a heart attack'`. This mirrors what real tokenisers do before splitting. Notice that the cleaner knows about **only two** punctuation marks: `.` and `?`. Remember this, because the Challenge is about the one it does not know.

---

**Step 3: Split into tokens.**

**`tokens = clean_text.split()`**

`split()` with no argument cuts the string wherever there is whitespace and returns a Python **list** of the pieces. Each piece is a token. For our example the list is:

`['my', 'chest', 'hurts', 'is', 'it', 'a', 'heart', 'attack']`

This is the simplest possible tokeniser. Real tokenisers also split *inside* words, but the principle (text in, pieces out) is the same.

**`token_array = np.array(tokens)`**

Converts the Python list into a **NumPy array**. Arrays store values of one type in a compact block of memory and support fast operations, which is what the later stages of an LLM pipeline need. The array's data type will be a fixed-width Unicode string sized to the longest token (`attack` has 6 characters).

---

**Step 4: Build a vocabulary.**

**`vocab, inverse_indices = np.unique(token_array, return_inverse=True)`**

This single line does two jobs. `np.unique` looks at the array and returns:

1. **`vocab`**: the **sorted** array of distinct tokens, with duplicates removed. This is the system's "dictionary".
2. **`inverse_indices`** (because we passed `return_inverse=True`): for **every token in the original array**, its position inside `vocab`. These positions are our **token IDs**.

A tiny illustration in words: if the tokens were `help, me, help, now`, the vocabulary would be `help, me, now` (sorted, no duplicates), and the IDs would be `0, 1, 0, 2`. Notice that `help` gets the **same ID both times**. That stable mapping is the whole point of a vocabulary: identical words always map to identical numbers.

**`return token_array, vocab, inverse_indices`**

Sends all three results back to whoever called the function.

---

## Step 5: Print and inspect

Now we use the function on a real patient case from the AfyaPlus portal. The code below stores the message in `patient_query`, calls the function, and unpacks the three returned values into `tokens`, `vocabulary` and `token_ids`. Then it prints everything.

Run the cell and read the output carefully.

In [ ]:
# Real patient case from the AfyaPlus portal
patient_query = "My chest hurts. Is it a heart attack?"
tokens, vocabulary, token_ids = simple_numpy_tokenizer(patient_query)

print("--- Tokenisation Results ---")
print(f"Original Text: {patient_query}")
print(f"Tokens: {tokens}")
print(f"Vocabulary: {vocabulary}")
print(f"Token IDs (Numerical Representation): {token_ids}")

### How to read the output

You should see this:

```
--- Tokenisation Results ---
Original Text: My chest hurts. Is it a heart attack?
Tokens: ['my' 'chest' 'hurts' 'is' 'it' 'a' 'heart' 'attack']
Vocabulary: ['a' 'attack' 'chest' 'heart' 'hurts' 'is' 'it' 'my']
Token IDs (Numerical Representation): [7 2 4 5 6 0 3 1]
```

**Tokens** are in the original order: eight of them.

**Vocabulary** is the same eight words, but now **sorted alphabetically**: `a` is position 0, `attack` is position 1, `chest` is position 2, and so on up to `my` at position 7.

**Token IDs** are the position of each token in that sorted vocabulary. Read them against the tokens one at a time:

| Token | Position in vocabulary | ID |
|---|---|---|
| my | 7 | 7 |
| chest | 2 | 2 |
| hurts | 4 | 4 |
| is | 5 | 5 |
| it | 6 | 6 |
| a | 0 | 0 |
| heart | 3 | 3 |
| attack | 1 | 1 |

That is why the IDs print as `[7 2 4 5 6 0 3 1]`.

**What to check:** confirm that identical words always map to the same ID. In this example no word repeats, so try changing the message in `patient_query` to include a repeated word (for example repeat `chest`) and run both cells again. The repeated word keeps one ID.

### How the code works (summary from the lesson)

The function turns text into a NumPy array of tokens, then derives a sorted vocabulary and the integer ID each token maps to. `np.unique` with `return_inverse` gives the same *numericalisation pattern* (token to integer) that production tokenisers like `tiktoken` produce, but with code we can read and modify.

### An important difference from production

In this toy tokeniser, the IDs depend on **which words are in the input**, because the vocabulary is rebuilt from the input every time. A production tokeniser has a **fixed vocabulary** learned once during training (around 100,000 entries), so the word `chest` gets the **same** ID in every message ever sent. Keep this in mind for Self-check Question 3.

### The limitation that leads to the Challenge

Look again at the cleaning line. It removes `.` and `?` only. If a panicked patient types `Help!!!`, the exclamation marks are **not** removed and **not** separated, so the whole of `help!!!` is one token. That is a different token from `help`, and the urgency signal is hidden inside it. You will fix this in the companion notebook, **Challenge: The AfyaPlus Emergency Tokeniser**.

---

## Did you run the code? Output self-check

These three questions build in depth: first **recall** what your code printed, then **apply** the idea by predicting how a change affects the output, then **analyse** why it behaves that way. If you cannot answer the first from memory, run the code again before continuing.

Try each question yourself before reading the answers at the bottom of this section.

**Question 1 (Level 1, Recall: what the code printed)**
Running the tokeniser on `'My chest hurts. Is it a heart attack?'` prints how many tokens?

- A. Eight tokens in the list
- B. Ten tokens in the list
- C. One token per letter
- D. Six tokens in the list

**Question 2 (Level 2, Apply: predict the change)**
If you changed the input to `'My chest hurts!!!'` without editing the cleaner, how many tokens print?

- A. Four; punctuation splits words
- B. Eleven; each `!` is a token
- C. Zero; the code errors
- D. Still eight; `!!!` is stripped

**Question 3 (Level 3, Analyse: reason about why)**
Two different inputs sharing no words could still get overlapping Token IDs. Why?

- A. IDs are random per run
- B. IDs index a shared sorted vocabulary
- C. IDs come from character codes
- D. IDs encode word meaning

*To test Question 2 yourself, change the text in the `patient_query` line in the Step 5 cell, then re-run it.*

### Answers and explanations

**Question 1: A (eight tokens).** After cleaning, the sentence is `my chest hurts is it a heart attack`. Splitting on spaces gives eight words, so eight tokens.

**Question 2: read this one carefully.** The cleaner removes only `.` and `?`, so `!!!` is **not** stripped and **not** split off. The text becomes `my chest hurts!!!`, and splitting on spaces gives three pieces: `my`, `chest`, `hurts!!!`. The code prints **three** tokens. None of the four options says three, so use this as a teaching moment: always run the code instead of trusting a prediction. The key lesson is that `hurts!!!` is one token, which is the exact problem the Challenge fixes.

*Facilitator note: check which option the course platform marks as correct for this question before the session, so you can address any confusion.*

**Question 3: B (IDs index a shared sorted vocabulary).** An ID is just a **position in a sorted list**. For example, the inputs `fever cough` and `rash itching` share no words, yet both produce the IDs `[1 0]`, because in each case the ID means "my position in *this* sorted list". IDs are not random (A), not character codes (C), and carry no meaning (D). Meaning arrives in the next lesson, **Embeddings**.

---

## Quick check: recall before you continue

**Why must text be converted to integer IDs before the model can use it?**

<details>
<summary><b>Reveal answer</b></summary>

A neural network is made of mathematical operations (multiplications and additions) that work on numbers, not on letters. Text has to be turned into numbers first. Integer IDs are a compact, stable way to refer to each token, and in the next stage each ID is looked up to get a vector of numbers (an embedding) that the model can compute with.

</details>

---

## Summary

| Step | What the code does | Key tool |
|---|---|---|
| 1 | Imports the toolkit | `import numpy as np` |
| 2 | Lowercases and removes `.` and `?` | `.lower()`, `.replace()` |
| 3 | Splits on spaces into tokens, stores as an array | `.split()`, `np.array()` |
| 4 | Finds distinct tokens and assigns IDs | `np.unique(..., return_inverse=True)` |
| 5 | Prints tokens, vocabulary and IDs | `print`, f-strings |

**Key ideas**

- Models read numbers, not words. Tokenisation is the bridge.
- The same token must always map to the same ID.
- Token count drives cost.
- This whitespace tokeniser is a teaching tool. Production uses BPE tokenisers such as `tiktoken`.

**Next:** open the companion notebook for the Challenge. When you finish, reply with **"Done"**.

*Coming up next: Embeddings. See how the model turns those numbered tokens into meaning it can reason over.*